In [39]:
from urllib import request
import torch
from torch import nn
import re

In [40]:
raw =  ""
urls = [
    # "http://www.gutenberg.org/files/2554/2554-0.txt", # Crime and Punishment
    "https://www.gutenberg.org/cache/epub/28054/pg28054.txt", # The Brothers Karamazov
    # "https://www.gutenberg.org/cache/epub/600/pg600.txt", # Notes from the underground
    # "https://www.gutenberg.org/cache/epub/2638/pg2638.txt", # The Idiot
    "https://www.gutenberg.org/cache/epub/59196/pg59196.txt", ## Collection of alot of books by Dostoevsky
    "https://www.gutenberg.org/cache/epub/40745/pg40745.txt", # Short stories by Dostoevsky

]

for url in urls:
    response = request.urlopen(url)
    raw = raw + " " + response.read().decode('utf8')

In [3]:
import re

def clean_text(text):
    text = text.lower()
    text = re.sub(r'[\n\t\r]+', ' ', text)
    text = re.sub(r' +', ' ', text)
    return text.strip()

raw = clean_text(raw)


In [18]:
chars = sorted(list(set(raw)))
vocab_size = len(chars)
print(f"Unique chars (vocab size): {vocab_size}")

stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for i, ch in enumerate(chars)}

encoded = [stoi[c] for c in raw]

split_ratio = 0.9
split_idx = int(len(encoded) * split_ratio)
train_data = encoded[:split_idx]
test_data = encoded[split_idx:]

print(f"Train data length: {len(train_data)}")
print(f"Test data length: {len(test_data)}")

seq_len = 50

Unique chars (vocab size): 78
Train data length: 2195325
Test data length: 243925


In [25]:
import torch

def get_sequences(data, seq_len, batch_size, stride=1):
    x_batches = []
    y_batches = []
    inputs = []
    targets = []

    for i in range(0, len(data) - seq_len - 1, stride):
        x = data[i : i + seq_len]
        y = data[i + 1 : i + seq_len + 1]
        inputs.append(x)
        targets.append(y)

        if len(inputs) == batch_size:
            batch_x = torch.tensor(inputs, dtype=torch.long)   # shape (batch_size, seq_len)
            batch_y = torch.tensor(targets, dtype=torch.long)  # shape (batch_size, seq_len)
            x_batches.append(batch_x)
            y_batches.append(batch_y)
            inputs = []
            targets = []

    if len(inputs) > 0:
        batch_x = torch.tensor(inputs, dtype=torch.long)
        batch_y = torch.tensor(targets, dtype=torch.long)
        x_batches.append(batch_x)
        y_batches.append(batch_y)

    return list(zip(x_batches, y_batches))


train_sequences = get_sequences(train_data, seq_len, 128)
test_sequences = get_sequences(test_data, seq_len, 128)

print(f"Number of train sequences: {len(train_sequences)}")
print(f"Number of test sequences: {len(test_sequences)}")

x_seq = torch.tensor(train_sequences[0][0], dtype=torch.long).unsqueeze(0)  # shape: (1, seq_len)
y_seq = torch.tensor(train_sequences[0][1], dtype=torch.long).unsqueeze(0)  # shape: (1, seq_len)
print(x_seq)
print(x_seq.shape)

Number of train sequences: 17151
Number of test sequences: 1906
tensor([[[77, 49, 37,  ..., 47, 30, 42],
         [49, 37, 34,  ..., 30, 42, 30],
         [37, 34,  0,  ..., 42, 30, 55],
         ...,
         [42, 44, 48,  ..., 49, 37,  0],
         [44, 48, 49,  ..., 37,  0, 30],
         [48, 49,  0,  ...,  0, 30, 41]]])
torch.Size([1, 128, 50])


/tmp/ipython-input-25-2193736817.py:40: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  x_seq = torch.tensor(train_sequences[0][0], dtype=torch.long).unsqueeze(0)  # shape: (1, seq_len)
/tmp/ipython-input-25-2193736817.py:41: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  y_seq = torch.tensor(train_sequences[0][1], dtype=torch.long).unsqueeze(0)  # shape: (1, seq_len)


### Implementation starts here

In [26]:
import torch
import torch.nn as nn
import torch.nn.functional as F

In [27]:
class RNNLayer(nn.Module):
  def __init__(self, input_size, hidden_size):
    super().__init__()
    self.W = nn.Parameter(torch.rand(input_size, hidden_size) * 0.01)
    self.H = nn.Parameter(torch.rand(hidden_size, hidden_size) * 0.01)
    self.b = nn.Parameter(torch.zeros(hidden_size))

  def forward(self, x, hidden):
    h_next = torch.tanh(x @ self.W + hidden @ self.H + self.b)
    return h_next

In [28]:
class SimpleRNN(nn.Module):
  def __init__(self, vocab_size, hidden_size):
    super().__init__()
    self.vocab_size = vocab_size
    self.hidden_size = hidden_size

    self.rnn_layer = RNNLayer(vocab_size, hidden_size)
    self.fc = nn.Linear(hidden_size, vocab_size)

  def forward(self, x, hidden = None):
    batch_size, seq_len = x.size()

    if hidden is None:
      hidden = torch.zeros(batch_size, self.hidden_size, device = x.device)

    outputs = []

    for t in range(seq_len):
      x_t_indices = x[:, t]

      x_t_one_hot = torch.zeros(batch_size, self.vocab_size, device=x.device)

      batch_idxs = torch.arange(batch_size, device=x.device)
      x_t_one_hot[batch_idxs, x_t_indices] = 1.0

      hidden = self.rnn_layer(x_t_one_hot, hidden)
      outputs.append(hidden.unsqueeze(1))

    outputs = torch.cat(outputs, dim = 1)
    logits = self.fc(outputs)

    return logits, hidden


In [29]:
def train_epoch(model, train_sequences, optimizer, criterion, device):
    model.train()
    total_loss = 0.0
    for x_seq, y_seq in train_sequences:
        x = torch.tensor(x_seq, dtype=torch.long).to(device)
        y = torch.tensor(y_seq, dtype=torch.long).to(device)

        optimizer.zero_grad()
        logits, _ = model(x)  # logits: (1, seq_len, vocab_size)

        loss = criterion(logits.view(-1, logits.size(-1)), y.view(-1))
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
    avg_loss = total_loss / len(train_sequences)
    return avg_loss


def evaluate(model, test_sequences, criterion, device):
    model.eval()
    total_loss = 0.0
    with torch.no_grad():
        for x_seq, y_seq in test_sequences:
            x = torch.tensor(x_seq, dtype=torch.long).to(device)
            y = torch.tensor(y_seq, dtype=torch.long).to(device)

            logits, _ = model(x)
            loss = criterion(logits.view(-1, logits.size(-1)), y.view(-1))
            total_loss += loss.item()
    avg_loss = total_loss / len(test_sequences)
    return avg_loss


In [31]:
model = SimpleRNN(vocab_size, 256)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

num_epochs = 5

for epoch in range(num_epochs):
    train_loss = train_epoch(model, train_sequences, optimizer, criterion, device)
    print(f"Epoch: {epoch} ---- Loss: {train_loss}")



/tmp/ipython-input-29-2225684716.py:5: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  x = torch.tensor(x_seq, dtype=torch.long).to(device)
/tmp/ipython-input-29-2225684716.py:6: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  y = torch.tensor(y_seq, dtype=torch.long).to(device)


Epoch: 0 ---- Loss: 1.7786190661527166
Epoch: 1 ---- Loss: 1.5221709319366519
Epoch: 2 ---- Loss: 1.4683843594439543
Epoch: 3 ---- Loss: 1.442631448853378
Epoch: 4 ---- Loss: 1.4261052283342153
Epoch: 5 ---- Loss: 1.4161865060985988
Epoch: 6 ---- Loss: 1.4084720475480033
Epoch: 7 ---- Loss: 1.402013527638752
Epoch: 8 ---- Loss: 1.3987086064599636
Epoch: 9 ---- Loss: 1.3948535039316328


In [32]:
test_loss = evaluate(model, test_sequences, criterion, device)
print(test_loss)

/tmp/ipython-input-29-2225684716.py:25: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  x = torch.tensor(x_seq, dtype=torch.long).to(device)
/tmp/ipython-input-29-2225684716.py:26: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  y = torch.tensor(y_seq, dtype=torch.long).to(device)


1.6129614667026844


In [33]:
import torch
import torch.nn.functional as F

def generate_text(model, start_sequence, gen_len=100, device='cuda'):
    model.eval()
    hidden = None

    input_ids = [stoi[c] for c in start_sequence]
    x = torch.as_tensor([input_ids], dtype=torch.long, device=device)
    with torch.no_grad():
        _, hidden = model(x)

    last_idx = input_ids[-1]
    generated = list(start_sequence)

    for _ in range(gen_len):
        x = torch.as_tensor([[last_idx]], dtype=torch.long, device=device)
        logits, hidden = model(x, hidden)
        logits = logits[:, -1, :]
        probs  = F.softmax(logits, dim=-1)

        next_idx = torch.multinomial(probs, num_samples=1).item()

        generated.append(itos[next_idx])
        last_idx = next_idx

    return ''.join(generated)


In [42]:
generate_text(model, "i came home")

"i came homeritents: he was wawhing ilussine keepiinately passes i'tess he was two very from dishote invonlember"